In [1]:
from pathlib import Path

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

### Local model load

In [2]:
MODEL_PATH = Path.home() / "rag-learning-practice" / "models" / "all-MiniLM-L6-v2"

In [3]:
# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

docs = [doc1, doc2, doc3, doc4, doc5]


In [4]:
# ২. Local embedding setup
embeddings = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={
        "device": "cpu",
        "local_files_only": True
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

# ৩. Local Chroma DB make and text save
vector_store = Chroma(
    collection_name="rag_learning",
    embedding_function=embeddings,
    persist_directory="my_chroma_db"
)

/home/mehedi-hassan/rag-learning-practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10628.94it/s]


In [5]:
# add documents
vector_store.add_documents(docs)

['6f6dd67d-b693-4c69-a3d6-dcbb4dc0aae0',
 '2e1a337f-14aa-4918-bb56-bbb2aaa8af86',
 'f7b878cc-404a-4d9b-971e-94de8563670c',
 'bea3dbf1-28d7-4577-b69b-a24f70cdc47e',
 'e302fc7e-71ee-44bd-881f-1396b876a737']

In [6]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['6f6dd67d-b693-4c69-a3d6-dcbb4dc0aae0',
  '2e1a337f-14aa-4918-bb56-bbb2aaa8af86',
  'f7b878cc-404a-4d9b-971e-94de8563670c',
  'bea3dbf1-28d7-4577-b69b-a24f70cdc47e',
  'e302fc7e-71ee-44bd-881f-1396b876a737'],
 'embeddings': array([[ 0.00994728,  0.06914341, -0.05147117, ..., -0.03543337,
          0.0128481 ,  0.01248289],
        [ 0.00127744,  0.03129854, -0.02375378, ..., -0.00518361,
         -0.03280617,  0.02737717],
        [-0.10265916,  0.02650811,  0.02271503, ..., -0.03359746,
         -0.07984947, -0.01507707],
        [ 0.02123393, -0.02468549, -0.04494375, ..., -0.10995814,
          0.00572561,  0.09915381],
        [ 0.01873977,  0.04382848, -0.04304256, ..., -0.07801621,
         -0.07840683, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [7]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k= 2
)

[Document(id='bea3dbf1-28d7-4577-b69b-a24f70cdc47e', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='2e1a337f-14aa-4918-bb56-bbb2aaa8af86', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [8]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='bea3dbf1-28d7-4577-b69b-a24f70cdc47e', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693601727485657),
 (Document(id='2e1a337f-14aa-4918-bb56-bbb2aaa8af86', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.1493451595306396)]

In [9]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="Who is the captain of Chennai Super Kings?",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='f7b878cc-404a-4d9b-971e-94de8563670c', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.6683117151260376),
 (Document(id='e302fc7e-71ee-44bd-881f-1396b876a737', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.9662648439407349)]

In [10]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='d26c7da3-4540-40c6-aea1-8b6a7b2cce57', document=updated_doc1)

In [11]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['6f6dd67d-b693-4c69-a3d6-dcbb4dc0aae0',
  '2e1a337f-14aa-4918-bb56-bbb2aaa8af86',
  'f7b878cc-404a-4d9b-971e-94de8563670c',
  'bea3dbf1-28d7-4577-b69b-a24f70cdc47e',
  'e302fc7e-71ee-44bd-881f-1396b876a737'],
 'embeddings': array([[ 0.00994728,  0.06914341, -0.05147117, ..., -0.03543337,
          0.0128481 ,  0.01248289],
        [ 0.00127744,  0.03129854, -0.02375378, ..., -0.00518361,
         -0.03280617,  0.02737717],
        [-0.10265916,  0.02650811,  0.02271503, ..., -0.03359746,
         -0.07984947, -0.01507707],
        [ 0.02123393, -0.02468549, -0.04494375, ..., -0.10995814,
          0.00572561,  0.09915381],
        [ 0.01873977,  0.04382848, -0.04304256, ..., -0.07801621,
         -0.07840683, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [12]:
# delete document
vector_store.delete(ids=['d26c7da3-4540-40c6-aea1-8b6a7b2cce57'])

In [13]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['6f6dd67d-b693-4c69-a3d6-dcbb4dc0aae0',
  '2e1a337f-14aa-4918-bb56-bbb2aaa8af86',
  'f7b878cc-404a-4d9b-971e-94de8563670c',
  'bea3dbf1-28d7-4577-b69b-a24f70cdc47e',
  'e302fc7e-71ee-44bd-881f-1396b876a737'],
 'embeddings': array([[ 0.00994728,  0.06914341, -0.05147117, ..., -0.03543337,
          0.0128481 ,  0.01248289],
        [ 0.00127744,  0.03129854, -0.02375378, ..., -0.00518361,
         -0.03280617,  0.02737717],
        [-0.10265916,  0.02650811,  0.02271503, ..., -0.03359746,
         -0.07984947, -0.01507707],
        [ 0.02123393, -0.02468549, -0.04494375, ..., -0.10995814,
          0.00572561,  0.09915381],
        [ 0.01873977,  0.04382848, -0.04304256, ..., -0.07801621,
         -0.07840683, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo